# Run project

**Ce notebook permet d'exécuter l'ensemble du pipeline du projet**

In [0]:
# Dans une cellule, vérifie que le fichier est accessible
dbutils.fs.ls("/Volumes/main/default/raw/")

In [0]:
import os
import sys
os.getcwd()
import os
os.chdir("/Workspace/Users/karl.sondeji@aivancity.education/Spark-pipeline-on-Online-Retail")
os.getcwd()

In [0]:
# Restart le kernel python
import importlib, sys
for k in list(sys.modules):
    if k.startswith("src."):
        del sys.modules[k]

In [0]:
from src.main import run_pipeline
run_pipeline(analytics=True)

In [0]:
from src.analytics.sales_analysis import get_sales_by_continent, get_sales_by_purchase_segment

gold = "/Volumes/main/default/raw/gold"
get_sales_by_continent(spark, gold).show()
get_sales_by_purchase_segment(spark, gold).show()

## Rapport complet (tables phase*, historique Delta, time travel)

Enregistre `main.default.phase1` … `phase4`, les tables partitionnées du notebook, puis affiche `DESCRIBE HISTORY` et les comparaisons `VERSION AS OF`.

In [0]:
from src.utils.config import get_config
from src.analytics.runner import run_full_analytics_report

config = get_config()
report = run_full_analytics_report(spark, config)

# Requêtes SQL directes (comme le notebook d'origine)
spark.sql("DESCRIBE HISTORY main.default.phase4").show(truncate=False)
spark.sql("SELECT COUNT(*) FROM main.default.phase4").show()
spark.sql("SHOW PARTITIONS main.default.sales_per_country_continent").show(20, truncate=False)

In [0]:
from src.analytics.sales_analysis import (
    get_sales_by_country,
    get_sales_by_product_category,
    get_sales_by_continent,
    get_top_category_for_big_customer,
)

gold = "/Volumes/main/default/raw/gold"

get_sales_by_country(spark, gold).orderBy("total_revenue", ascending=False).show(20)
get_sales_by_product_category(spark, gold).show()
get_sales_by_continent(spark, gold).show()
get_top_category_for_big_customer(spark, gold).show()

In [0]:
display(spark.read.format("delta").load("/Volumes/main/default/raw/gold"))